In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Merged file.csv")
df.columns = df.columns.str.strip()
df

,Age,Occupation,Assets,Investments,Savings,Debt,Gross monthly income,Net monthly income,Rent/Mortgage,Utilities,...,Clothes,Phone,Subscriptions,Miscellaneous,Vacations,Gifts,Emergency Fund,Dining out,Movies,Other
0,30,Salaried,2550000,160000,42000,31000,51000,43000,6100,2100,...,1350,720,570,1150,2600,820,12200,2100,520,360
1,45,Homemaker,2650000,130000,62000,0,37000,33000,0,2350,...,1100,740,0,1080,1600,840,7200,1270,440,0
2,21,Student,90000,18000,8000,15000,13000,12000,3500,700,...,600,350,160,250,400,250,800,500,160,0
3,25,Salaried,2300000,130000,36000,54000,44000,37000,8500,1970,...,1260,690,540,1060,2060,760,10600,1860,440,340
4,29,Salaried,4400000,200000,51000,77000,66000,58000,12300,2550,...,2000,920,700,1500,5000,1500,20200,4100,720,500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,36,Salaried,4150000,365000,73000,41500,86500,76500,9150,2775,...,2175,1030,930,1430,3575,1130,18300,3275,830,465
144,35,Self-Employed,8400000,530000,160000,0,98000,85000,0,3700,...,3200,1300,1150,1900,8300,2600,37000,7300,1260,830
145,25,Salaried,2300000,130000,36000,54000,44000,37000,8500,1970,...,1260,690,540,1060,2060,760,10600,1860,440,340
146,53,Retired,2250000,125000,56000,0,32500,29500,0,2250,...,970,560,0,990,1220,560,5300,890,230,0


In [10]:
# Calculate Total Expenses
# Define expense columns (only if they exist in your data)
expense_cols = [
    "Rent/Mortgage", "Utilities", "Insurance", "Car Payment",
    "Debt Payments", "Groceries", "Clothes", "Phone",
    "Subscriptions", "Miscellaneous", "Vacations", "Gifts",
    "Dining out", "Movies", "Other"
]

# Filter for valid ones only
valid_expenses = [col for col in expense_cols if col in df.columns]

# Convert and sum
for col in valid_expenses:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '').str.strip(), errors='coerce')

df["Total Expenses"] = df[valid_expenses].sum(axis=1)

# Preview result
print(df[["Total Expenses"]].head())

   Total Expenses
0           28160
1           17320
2           10170
3           28500
4           45360


In [31]:
# Columns to convert (check they exist first)
columns_to_convert = [
    "Assets", "Investments", "Savings", "Debt",
    "Gross monthly income", "Net monthly income", "Emergency Fund", "Total Expenses"
]

for col in columns_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '').str.strip(),
            errors='coerce'
        )
# Drop rows with missing values
df.dropna(inplace=True)



In [12]:
# Generate Labels (Recommended Investment)
def recommend_investment(row):
    income = row["Gross monthly income"]
    savings = row["Savings"]
    debt = row["Debt"]
    emergency_fund = row["Emergency Fund"]
    expenses = row["Total Expenses"]
    occupation = row.get("Occupation", "").strip()

    if savings > 200000 and income > 80000 and debt < 50000:
        return "ELSS"
    elif savings > 100000 and debt < 50000:
        return "SIP"
    elif income < 40000 or occupation in ["Student", "Senior Citizen"]:
        return "FD"
    elif emergency_fund < (expenses * 3):
        return "PPF"
    else:
        return "Mixed"

df["Recommended Investment"] = df.apply(recommend_investment, axis=1)

In [13]:
# Prepare Features and Target
X = df[[
    "Gross monthly income", "Net monthly income", "Savings",
    "Investments", "Debt", "Emergency Fund", "Total Expenses"
]]
y = df["Recommended Investment"]

In [14]:
#Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Train Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [17]:
# Evaluate Model
y_pred = model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

          FD       1.00      1.00      1.00         8
         PPF       0.94      1.00      0.97        17
         SIP       1.00      0.80      0.89         5

    accuracy                           0.97        30
   macro avg       0.98      0.93      0.95        30
weighted avg       0.97      0.97      0.97        30


Confusion Matrix:
 [[ 8  0  0]
 [ 0 17  0]
 [ 0  1  4]]


**Model**

In [18]:
# Save the Model
joblib.dump(model, "investment_model.pkl")

['investment_model.pkl']

**Investment Recommendation Function**

In [29]:
import pandas as pd
import joblib

# Load the trained investment model
model = joblib.load("investment_model.pkl")

def recommend_investment_strategy(user_data: dict):
    """
    Predicts investment strategy and provides personalized advice.
    """
    # Convert to DataFrame
    df = pd.DataFrame([user_data])
    
    # Predict investment strategy
    prediction = model.predict(df)[0]
    
    # Extract relevant data
    income = user_data["Gross monthly income"]
    savings = user_data["Savings"]
    debt = user_data["Debt"]
    emergency_fund = user_data["Emergency Fund"]
    expenses = user_data["Total Expenses"]
    
    # Generate financial advice
    advice = []
    
    if emergency_fund < expenses * 3:
        advice.append(f" Your emergency fund is below the 3-month threshold. Aim for ₹{expenses*3:,.0f}.")
    
    if savings < 100000:
        advice.append(" Your savings are on the lower side. Consider increasing them before aggressive investing.")
    
    if prediction == "ELSS":
        advice.append(" ELSS is suitable for high-income individuals comfortable with long lock-in and equity risk.")
    elif prediction == "SIP":
        advice.append(" SIP is a smart choice for steady long-term wealth building with moderate risk.")
    elif prediction == "FD":
        advice.append(" FD is a safe option if you're risk-averse or need fixed returns.")
    elif prediction == "PPF":
        advice.append(" PPF is good when emergency funds are low and you need tax-saving, long-term safety.")
    elif prediction == "Mixed":
        advice.append(" A mixed strategy is ideal when no single option dominates — diversify across SIP, FD, and ELSS.")
    
    # Display results
    print(" Input Summary:")
    for key, value in user_data.items():
        print(f"  - {key}: ₹{value:,}")
    
    print("\n Recommended Investment Strategy:", prediction)
    print("\n Financial Advice:")
    for line in advice:
        print("-", line)
    
    return prediction, advice


**Testing example**

In [30]:
user = {
    "Gross monthly income": 75000,
    "Net monthly income": 62000,
    "Savings": 180000,
    "Investments": 40000,
    "Debt": 25000,
    "Emergency Fund": 50000,
    "Total Expenses": 40000
}

recommend_for = recommend_investment_strategy(user)


 Input Summary:
  - Gross monthly income: ₹75,000
  - Net monthly income: ₹62,000
  - Savings: ₹180,000
  - Investments: ₹40,000
  - Debt: ₹25,000
  - Emergency Fund: ₹50,000
  - Total Expenses: ₹40,000

 Recommended Investment Strategy: PPF

 Financial Advice:
-  Your emergency fund is below the 3-month threshold. Aim for ₹120,000.
-  PPF is good when emergency funds are low and you need tax-saving, long-term safety.
